# 01 - Scrape Tanqeeb Job Listings

This notebook is the **scraping** stage of the Tanqeeb pipeline.

- Collects job listing URLs from the Tanqeeb search results (via Selenium, since results are paginated behind JS rendering), starting from page 1 each run and stopping early once it catches up with jobs already scraped.
- Scrapes each individual **new** job page (via `requests` + `BeautifulSoup`) for the relevant fields.
- Saves the full raw history to `tanqeeb_jobs_raw.json` (for auditing), and pushes just this run's new jobs onto a **pending queue** (`tanqeeb_pending_raw_queue.json`) for `02_clean_Tanqeeb.ipynb` to pick up. The queue only grows if cleaning hasn't run yet -- nothing is lost between runs.

Checkpoint files are used throughout so a crash never costs you all the work done so far.


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import TimeoutException, WebDriverException
from urllib3.exceptions import ReadTimeoutError
from urllib.parse import urljoin
from datetime import datetime
import time
import os
import json
from azure.identity import DefaultAzureCredential
from azure.storage.filedatalake import DataLakeServiceClient


In [ ]:
# ============================================================
# CONFIG
# ============================================================

BASE_URL = "https://saudi.tanqeeb.com"

SEARCH_PATH = "https://saudi.tanqeeb.com/jobs/search"

QUERY_STRING = (
    "keywords=&country=54&state=0&category=-1&"
    "workplace=0&search_period=0&lang=all&change_lang=1"
)

SOURCE_NAME = "Tanqeeb"
STORAGE_ACCOUNT = "jobpipelinedatalake"

RAW_PATH = f"abfss://raw@{STORAGE_ACCOUNT}.dfs.core.windows.net/tanqeeb"

CHECKPOINT_PATH = f"abfss://checkpoints@{STORAGE_ACCOUNT}.dfs.core.windows.net/tanqeeb"

RAW_JSON_OUTPUT = f"{RAW_PATH}/tanqeeb_jobs_raw.json"

JOBS_CHECKPOINT = f"{CHECKPOINT_PATH}/tanqeeb_jobs_checkpoint.json"

NEW_URLS_CHECKPOINT = f"{CHECKPOINT_PATH}/tanqeeb_new_urls_checkpoint.json"

PENDING_RAW_QUEUE = f"{CHECKPOINT_PATH}/tanqeeb_pending_raw_queue.json"



##################3

RAW_CONTAINER = "raw"
CHECKPOINT_CONTAINER = "checkpoints"

TANQEEB_RAW_PATH = "tanqeeb/tanqeeb_jobs_raw.json"

JOBS_CHECKPOINT_PATH = "tanqeeb/tanqeeb_jobs_checkpoint.json"
NEW_URLS_CHECKPOINT_PATH = "tanqeeb/tanqeeb_new_urls_checkpoint.json"
PENDING_RAW_QUEUE_PATH = "tanqeeb/tanqeeb_pending_raw_queue.json"
####################3


# How often (in newly scraped jobs) to flush progress to disk
JOBS_SAVE_EVERY = 50

# ---- Incremental scraping ----
# If True: pagination always starts from page 1 (newest listings first)
# and only job URLs NOT already in JOBS_CHECKPOINT are queued for
# scraping; pagination stops early once it looks like we've caught up
# with previously scraped jobs, so re-running this notebook only fetches
# jobs posted since the last run.
# If False: falls back to the original behavior -- every page is walked
# and every listing found is (re-)scraped.
INCREMENTAL_MODE = True

# Stop pagination once this many CONSECUTIVE pages contain zero new job
# URLs (i.e. every job on them was already scraped in a previous run).
# A couple of pages of buffer guards against occasional pinned/duplicate
# listings interrupting an otherwise "all new" or "all old" run.
# Only used when INCREMENTAL_MODE is True.
STOP_AFTER_CONSECUTIVE_KNOWN_PAGES = 2

# Safety cap in case pagination never signals "end" cleanly.
# ~50,700 jobs / ~28 per page is roughly 1,800+ pages, so raise
# this if you actually intend to (re)scrape the entire site.
MAX_PAGES = 2000

# Stop pagination after this many consecutive driver failures in a row
MAX_CONSECUTIVE_FAILURES = 5


In [ ]:
# AZURE DATA LAKE CLIENT

credential = DefaultAzureCredential()

service_client = DataLakeServiceClient(
    account_url=f"https://{STORAGE_ACCOUNT}.dfs.core.windows.net",
    credential=credential
)

In [ ]:
# ============================================================
# ADLS JSON HELPERS
# ============================================================

def get_file_client(container_name, file_path):
    """
    Return an ADLS Gen2 file client.
    """
    file_system = service_client.get_file_system_client(container_name)

    return file_system.get_file_client(file_path)


def adls_file_exists(container_name, file_path):
    """
    Check whether a file exists in ADLS.
    """
    file_client = get_file_client(container_name, file_path)

    try:
        file_client.get_file_properties()
        return True
    except Exception:
        return False


def read_json_from_adls(container_name, file_path, default=None):
    """
    Read a JSON file from ADLS.
    Returns default if the file does not exist.
    """
    file_client = get_file_client(container_name, file_path)

    try:
        download = file_client.download_file()
        content = download.readall()

        return json.loads(content.decode("utf-8"))

    except Exception:
        return default


def write_json_to_adls(container_name, file_path, data):
    """
    Write JSON data to ADLS, replacing the existing file.
    """
    file_client = get_file_client(container_name, file_path)

    content = json.dumps(
        data,
        ensure_ascii=False,
        indent=2
    )

    file_client.upload_data(
        content.encode("utf-8"),
        overwrite=True
    )


def delete_file_from_adls(container_name, file_path):
    """
    Delete a file from ADLS if it exists.
    """
    file_client = get_file_client(container_name, file_path)

    try:
        file_client.delete_file()
    except Exception:
        pass

In [ ]:
# ============================================================
# HELPER FUNCTIONS
# ============================================================

def format_datetime(dt):
    """
    Format a datetime as e.g. '11/9/2026 4:54:00 PM'
    (no leading zeros on day/month/hour, cross-platform).
    """
    if dt is None or pd.isna(dt):
        return "N/A"

    hour_12 = dt.hour % 12
    if hour_12 == 0:
        hour_12 = 12

    am_pm = "AM" if dt.hour < 12 else "PM"

    return f"{dt.day}/{dt.month}/{dt.year} {hour_12}:{dt.minute:02d}:{dt.second:02d} {am_pm}"


def build_page_url(page_num):
    """
    Tanqeeb paginates using a path segment, not a query param:
      Page 1: https://saudi.tanqeeb.com/jobs/search?keywords=...
      Page 2: https://saudi.tanqeeb.com/jobs/search/page/2?keywords=...
      Page N: https://saudi.tanqeeb.com/jobs/search/page/N?keywords=...
    """
    if page_num <= 1:
        return f"{SEARCH_PATH}?{QUERY_STRING}"
    return f"{SEARCH_PATH}/page/{page_num}?{QUERY_STRING}"


def get_job_links_from_html(html):
    soup = BeautifulSoup(html, "html.parser")
    return soup.select("h2.search-job-title a.search-job-title-link")


def make_driver():
    """Create a fresh headless Chrome driver with a longer command timeout."""
    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--disable-blink-features=AutomationControlled")
    d = webdriver.Chrome(options=options)
    # Default Selenium command timeout is 120s; bump it since some
    # search pages render slowly under headless Chrome after many navigations.
    d.command_executor.set_timeout(300)
    return d


def load_jobs_checkpoint():
    """
    Load the persistent master list of every job scraped so far.
    """
    return read_json_from_adls(
        CHECKPOINT_CONTAINER,
        JOBS_CHECKPOINT_PATH,
        default=[]
    )


def save_jobs_checkpoint(jobs):
    """
    Save the persistent master job list to ADLS.
    """
    write_json_to_adls(
        CHECKPOINT_CONTAINER,
        JOBS_CHECKPOINT_PATH,
        jobs
    )


def load_known_job_urls(jobs):
    """Set of job URLs already present in the master job list."""
    return {
        job.get("job_url")
        for job in jobs
        if job.get("job_url") and job.get("job_url") != "N/A"
    }


def load_new_urls_checkpoint():
    """
    Resume an interrupted discovery/scrape run.
    """
    data = read_json_from_adls(
        CHECKPOINT_CONTAINER,
        NEW_URLS_CHECKPOINT_PATH,
        default={"new_urls": []}
    )

    return data.get("new_urls", [])


def save_new_urls_checkpoint(new_urls):
    """
    Save newly discovered URLs to ADLS.
    """
    write_json_to_adls(
        CHECKPOINT_CONTAINER,
        NEW_URLS_CHECKPOINT_PATH,
        {
            "new_urls": new_urls
        }
    )


def clear_new_urls_checkpoint():
    """
    Remove the temporary discovery checkpoint.
    """
    delete_file_from_adls(
        CHECKPOINT_CONTAINER,
        NEW_URLS_CHECKPOINT_PATH
    )



def load_json_list(container_name, file_path):
    """
    Load a JSON list from ADLS.
    """
    return read_json_from_adls(
        container_name,
        file_path,
        default=[]
    )


def save_json_list(container_name, file_path, items):
    """
    Save a JSON list to ADLS.
    """
    write_json_to_adls(
        container_name,
        file_path,
        items
    )

## Step 1: Load what we already have, then discover NEW job URLs

In [ ]:
# Master list of every job ever scraped (persists across runs)
all_jobs = load_jobs_checkpoint()
known_job_urls = load_known_job_urls(all_jobs)

print(f"Jobs already in the dataset: {len(all_jobs)}")

if INCREMENTAL_MODE:
    print("Incremental mode ON: pagination starts from page 1 and only "
          "newly posted jobs will be queued for scraping.")
else:
    print("Incremental mode OFF: doing a full scrape of every listing found.")

# Resume an in-progress discovery run if one was interrupted (crash-safety),
# otherwise start pagination fresh from page 1 -- newest listings are shown
# first, so page 1 is where new postings will appear.
new_job_urls = load_new_urls_checkpoint()

if new_job_urls:
    print(f"Resuming an interrupted run: {len(new_job_urls)} new URL(s) "
          f"already discovered and pending scrape.")

driver = make_driver()
consecutive_failures = 0
consecutive_known_pages = 0
page_num = 1

try:
    while page_num <= MAX_PAGES:

        page_url = build_page_url(page_num)

        print(f"\nLoading search page {page_num}: {page_url}")

        try:
            driver.get(page_url)
            # Wait for page to load / JS to render results
            time.sleep(5)
            html = driver.page_source
            consecutive_failures = 0
        except (TimeoutException, WebDriverException, ReadTimeoutError) as e:
            consecutive_failures += 1
            print(f"  !! Driver error on page {page_num} "
                  f"(failure {consecutive_failures}/{MAX_CONSECUTIVE_FAILURES}): {e}")

            # Kill and restart the browser process; retry the same page
            try:
                driver.quit()
            except Exception:
                pass
            time.sleep(3)
            driver = make_driver()

            if consecutive_failures >= MAX_CONSECUTIVE_FAILURES:
                print("Too many consecutive driver failures. Stopping pagination "
                      "(progress so far is saved).")
                break

            continue  # retry page_num with the fresh driver, don't advance

        job_links = get_job_links_from_html(html)

        print(f"  -> Job links found on this page: {len(job_links)}")

        # Stop condition: no job cards on this page means we've
        # gone past the last page of results.
        if not job_links:
            print("No more job listings found. Stopping pagination.")
            break

        page_had_new_url = False
        new_on_this_page = 0

        for link in job_links:
            href = link.get("href")
            if not href:
                continue

            full_url = urljoin(BASE_URL, href)
            already_known = full_url in known_job_urls
            already_queued = full_url in new_job_urls

            if already_known and INCREMENTAL_MODE:
                # Already scraped in a previous run -- nothing to do.
                continue

            if not already_queued:
                new_job_urls.append(full_url)
                page_had_new_url = True
                new_on_this_page += 1

        if INCREMENTAL_MODE:
            if new_on_this_page == 0:
                consecutive_known_pages += 1
                print(f"  -> No new jobs on this page "
                      f"({consecutive_known_pages}/{STOP_AFTER_CONSECUTIVE_KNOWN_PAGES} "
                      f"consecutive already-known pages).")
                if consecutive_known_pages >= STOP_AFTER_CONSECUTIVE_KNOWN_PAGES:
                    print("Caught up with previously scraped jobs. Stopping pagination.")
                    break
            else:
                consecutive_known_pages = 0
                print(f"  -> New jobs found on this page: {new_on_this_page}")
        else:
            # Original (non-incremental) safety check: if a page returns
            # only links we've already queued (e.g. the site clamped an
            # out-of-range page back to the last page), treat that as the end.
            if not page_had_new_url:
                print("No new job URLs on this page (likely repeated last page). Stopping.")
                break

        # Persist discovery progress after every page
        save_new_urls_checkpoint(new_job_urls)

        page_num += 1

finally:
    driver.quit()

print(f"\nNewly discovered job URLs to scrape this run: {len(new_job_urls)}")


## Step 2: Scrape each newly discovered job page

In [ ]:
# Guard against re-scraping anything that (somehow) is already in the
# master list, and track how many genuinely new jobs this run adds.
already_scraped_urls = {job.get("job_url") for job in all_jobs}
jobs_added_this_run = 0
new_jobs_this_run = []  # only the jobs added in THIS run, for handoff to 02_clean

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/140.0.0.0 Safari/537.36"
    )
}

for i, url in enumerate(new_job_urls, start=1):

    if url in already_scraped_urls:
        continue

    print(f"\n[{i}/{len(new_job_urls)}] Scraping:")
    print(url)

    try:
        response = requests.get(url, headers=headers, timeout=30)

        if response.status_code != 200:
            print("Failed:", response.status_code)
            continue

        soup = BeautifulSoup(response.text, "html.parser")

        # JOB ID
        apply_button = soup.select_one(".apply-btn")
        job_id = apply_button.get("data-job-id") if apply_button else None

        # JOB TITLE
        title_element = soup.select_one("h3.job-title-text")
        job_title = title_element.get_text(" ", strip=True) if title_element else None

        # COMPANY NAME
        company_element = soup.select_one(
            "a.job-meta-company, .job-company-name, .company-name, a.company-name-link"
        )
        company_name = (
            company_element.get_text(" ", strip=True)
            if company_element else None
        )

        # WORK MODE + JOB TYPE
        tags = soup.select(".job-tags .job-tag")
        work_mode = None
        job_type = None

        for tag in tags:
            value = tag.get_text(" ", strip=True)
            value_lower = value.lower()

            if value_lower in ["on-site", "remote", "hybrid"]:
                work_mode = value
            elif value_lower in [
                "full time", "part time", "contract",
                "internship", "temporary", "freelance"
            ]:
                job_type = value

        # LOCATION
        location_element = soup.select_one(".job-meta-item span")
        country = None
        city = None

        if location_element:
            location_text = location_element.get_text(" ", strip=True)
            parts = [x.strip() for x in location_text.split(",")]

            if len(parts) >= 1:
                country = parts[0]
            if len(parts) >= 2:
                city = parts[1]

        # POSTED DATE
        date_element = soup.select_one(".job-date")
        raw_posted_date = date_element.get("data-datetime") if date_element else None

        parsed_posted_date = pd.to_datetime(raw_posted_date, errors="coerce") if raw_posted_date else None
        posted_date = format_datetime(parsed_posted_date) if parsed_posted_date is not None else "N/A"

        # JOB DETAILS
        job_details = {}
        meta_rows = soup.select(".meta-data .meta")

        for row in meta_rows:
            spans = row.find_all("span")
            if len(spans) >= 2:
                label = spans[0].get_text(" ", strip=True)
                value = spans[1].get_text(" ", strip=True)
                job_details[label] = value

        experience = job_details.get("Experience")
        employment = job_details.get("Employment")
        gender = job_details.get("Gender")
        education = job_details.get("Education")
        industry = job_details.get("Industry")
        career_level = job_details.get("Career Level")
        salary = job_details.get("Salary")

        # SKILLS
        skill_elements = soup.select(".job-skills .skill, .skills-list .skill-tag")
        skills_list = [
            s.get_text(" ", strip=True)
            for s in skill_elements
            if s.get_text(strip=True)
        ]
        skills = "; ".join(skills_list) if skills_list else None

        # JOB DESCRIPTION
        description_element = soup.select_one("#jobDescriptionBody")
        description = (
            description_element.get_text(" ", strip=True)
            if description_element else None
        )

        # SAVE JOB (raw fields as scraped)
        job = {
            "job_id": job_id if job_id else "N/A",
            "source": SOURCE_NAME,
            "job_title": job_title if job_title else "N/A",
            "company_name": company_name if company_name else "N/A",
            "job_description": description if description else "N/A",
            "job_url": url if url else "N/A",
            "country": country if country else "N/A",
            "city": city if city else "N/A",
            "work_mode": work_mode if work_mode else "N/A",
            "employment_type": employment if employment else (job_type if job_type else "N/A"),
            "career_level": career_level if career_level else "N/A",
            "experience_needed": experience if experience else "N/A",
            "qualification_required": education if education else "N/A",
            "skills": skills if skills else "N/A",
            "salary": salary if salary else "N/A",
            "estimated_salary": "",
            "posted_date": posted_date if posted_date else "N/A",
            "collected_at": format_datetime(datetime.now()),
        }

        all_jobs.append(job)
        new_jobs_this_run.append(job)
        already_scraped_urls.add(url)
        jobs_added_this_run += 1

        print("Scraped:", job_title)

        # Periodically flush progress so a crash doesn't lose everything
        if jobs_added_this_run % JOBS_SAVE_EVERY == 0:
            save_jobs_checkpoint(all_jobs)
            print(f"  (checkpoint saved: {len(all_jobs)} jobs total, "
                  f"{jobs_added_this_run} new this run)")

    except Exception as e:
        print("Error:", e)

    # Wait between requests
    time.sleep(1)

# Final checkpoint save after the loop completes. Every discovered URL has
# now been handled (scraped or errored), so clear the discovery checkpoint
# for next time.
save_jobs_checkpoint(all_jobs)
clear_new_urls_checkpoint()

print(f"\nNew jobs added this run: {jobs_added_this_run}")
print(f"Total jobs in dataset: {len(all_jobs)}")


## Step 3: Save raw data -- full history, plus a queue of only the NEW jobs

In [ ]:
# 1) Full historical raw mirror (every job ever scraped) -- useful for
#    auditing/debugging, not used directly by the rest of the pipeline.
save_json_list(
    RAW_CONTAINER,
    TANQEEB_RAW_PATH,
    all_jobs
)
# 2) Hand off ONLY this run's new jobs to 02_clean_Tanqeeb.ipynb via a
#    pending queue. We APPEND (not overwrite) so nothing is lost if
#    02_clean hasn't been run since the last scrape.
pending_raw = load_json_list(
    CHECKPOINT_CONTAINER,
    PENDING_RAW_QUEUE_PATH
)
pending_urls = {j.get("job_url") for j in pending_raw}

for job in new_jobs_this_run:
    if job.get("job_url") not in pending_urls:
        pending_raw.append(job)
        pending_urls.add(job.get("job_url"))

save_json_list(
    CHECKPOINT_CONTAINER,
    PENDING_RAW_QUEUE_PATH,
    pending_raw
)
# print("\n" + "=" * 60)
# print("New jobs found this run:", jobs_added_this_run)
# print("Total jobs in full raw history:", len(all_jobs))
# print("Jobs now waiting to be cleaned (pending queue):", len(pending_raw))
# print("Raw history saved as:", RAW_JSON_OUTPUT)
# print("Pending queue saved as:", PENDING_RAW_QUEUE)
# print("=" * 60)
